執行sql/01_data_cleaning.sql前需先在 BigQuery 建立目標 dataset

In [ ]:
CREATE SCHEMA IF NOT EXISTS traffic_ad_roi_clean;

| 步驟                           | 目的                           |
| ---------------------------- | ---------------------------- |
| Step 1 — Validation Checks   | 執行前先檢查，輸出 QA 報表              |
| Step 2 — Clean & Standardise | 清洗後寫入 traffic_ad_roi_clean.* |
| Step 3 — Summary Report      | 對比 raw vs clean 行數，確認結果      |

清洗邏輯說明
每張表均涵蓋以下清洗處理：

去重：ROW_NUMBER() OVER (PARTITION BY <pk>) 保留最新一筆，處理重複 primary key

NULL / 空值處理：COALESCE + NULLIF(TRIM(...), '') 統一填補預設值

Channel 標準化：CASE UPPER(TRIM(channel)) 將 GOOGLE、GOOGLE ADS 等變體統一為受控詞彙，對應你的 campaigns 表原有值

負數數值修正：GREATEST(..., 0) 防止 impressions、clicks、spend_usd 出現負數

CTR 重算：從原始 clicks/impressions 重新計算，比直接信任儲存值更可靠

衍生欄位：新增 is_active（campaigns）、cost_per_click_usd（ad_impressions）、engagement_tier（sessions）、order_value_tier（conversions）方便下游分析

***


## 資料規模總覽

| 表名 | 行數 | 類型 |
|---|---|---|
| `sessions` | **511,797** | 事實表（最大） |
| `conversions` | 18,288 | 事實表 |
| `ad_impressions` | 3,720 | 事實表 |
| `campaigns` | **12** | 維度表（只有12個活動） |
| `v_campaign_daily_ctr_cvr` | 2,988 | 每日時序 view |
| `v_monthly_channel_trend` | 60 | 月度 view |
| `v_monthly_roi_trend` | 36 | 月度 ROI view |
| `v_campaign_roi` | 12 | 活動彙總 view |
| `v_campaign_ctr_cvr_scatter` | 10 | 散點圖 view |
| `v_campaign_type_roi` | 11 | 類型彙總 view |
| `v_channel_performance` | 5 | 渠道彙總 view |
| `v_device_channel_conversion` | 15 | 裝置×渠道 view |
| `v_ctr_bucket_analysis` | 6 | CTR 分桶 view |



***

## 12 個 Campaign 分析

資料涵蓋 2024 年全年，共 5 個渠道、12 個活動 ：

| 渠道 | 活動數 | 預算範圍/日 |
|---|---|---|
| Google Ads | 4 | $300–$800 |
| Facebook Ads | 3 | $350–$700 |
| Email | 3 | $30–$80 |
| Organic | 1 | $0 |
| Direct | 1 | $0 |

***

## 關鍵業務洞察（來自 v_campaign_roi & v_channel_performance）

### ROAS 排名（由高至低）

| 活動 | 渠道 | ROAS | ROI % |
|---|---|---|---|
| Email_Abandoned_Cart | Email | **45.12** | 4,412% |
| Email_Newsletter_Monthly | Email | 27.86 | 2,686% |
| Email_Promo_Flash_Sale | Email | 26.95 | 2,595% |
| Google_Shopping_Q1 | Google Ads | 3.54 | 254% |
| Facebook_Retargeting | Facebook Ads | 2.63 | 163% |
| Google_Nonbrand_Search | Google Ads | 2.17 | 117% |
| Google_Brand_Search | Google Ads | 2.12 | 112% |
| Facebook_Awareness | Facebook Ads | 1.08 | 8% |
| Facebook_Conversion | Facebook Ads | 1.04 | 4% |
| **Google_Display_Remarketing** | Google Ads | **0.70** | **-30%** ⚠️ |

> ⚠️ `Google_Display_Remarketing` 是唯一**虧損活動**（ROI -30%），值得在 Power BI 重點標示。

### 渠道效率對比 

| 渠道 | ROAS | CVR | CPA |
|---|---|---|---|
| Email | **30.8** | 7.4% | $3.36 |
| Google Ads | 2.15 | 3.5% | $44.45 |
| Facebook Ads | 1.38 | 2.7% | $64.08 |
| Direct | N/A | 6.8% | $0 |
| Organic | N/A | 1.9% | $0 |

***

## Power BI 報告頁面建議

基於以上數據，建議設計 **5 個報告頁面**：

| 頁面 | 主視覺 | 使用的表 |
|---|---|---|
| **1. Executive Summary** | KPI Cards (Revenue, ROAS, CVR, CPA) + 渠道 Donut | `v_channel_performance` |
| **2. Campaign ROI** | Bar chart ROAS排名 + 虧損警示 + Table | `v_campaign_roi` |
| **3. Channel Trend** | 月度折線圖 (Revenue + Spend) | `v_monthly_roi_trend`, `v_monthly_channel_trend` |
| **4. CTR & CVR 分析** | 散點圖 CTR vs CVR + CTR 分桶 Bar | `v_campaign_ctr_cvr_scatter`, `v_ctr_bucket_analysis` |
| **5. Device & Country** | Matrix 裝置×渠道 + 地圖/Bar | `v_device_channel_conversion`, `sessions` |

***

## Power BI Import 策略

- **Import 模式**：`campaigns`（12行，維度）、`v_campaign_roi`、`v_channel_performance`、`v_monthly_roi_trend`、`v_monthly_channel_trend`、`v_campaign_ctr_cvr_scatter`、`v_ctr_bucket_analysis`、`v_device_channel_conversion`、`v_campaign_type_roi`
- **視情況 Import**：`ad_impressions`（3,720行）、`conversions`（18,288行）
- **謹慎考慮**：`sessions`（511,797行）— 建議在 BigQuery 先聚合再 import，避免 Power BI 檔案過大



`sessions` 有 511,797 行，直接 import 會令 `.pbix` 很大且慢。以下是針對你的報告需求，預先在 BigQuery 聚合的 SQL。

## 需要哪些聚合維度？

根據 5 個報告頁面，`sessions` 主要用於：
- 渠道 × 裝置 × 國家的流量分佈
- 每月趨勢
- Bounce rate 和 engagement tier

***

## 建議建立 3 個聚合 View

### View 1：每日 × 渠道 × 裝置（替代原始 sessions）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_daily_summary` AS

SELECT
  session_date,
  channel,
  device,
  country,
  engagement_tier,
  COUNT(*)                              AS total_sessions,
  COUNTIF(is_bounce = 1)               AS bounced_sessions,
  SUM(pages_viewed)                    AS total_pages_viewed,
  SUM(session_duration_sec)            AS total_duration_sec,
  COUNTIF(engagement_tier = 'High')    AS high_engagement_sessions,
  COUNTIF(engagement_tier = 'Medium')  AS medium_engagement_sessions,
  COUNTIF(engagement_tier = 'Low')     AS low_engagement_sessions
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY
  session_date, channel, device, country, engagement_tier;



> 511,797 行 → 預計壓縮至約 **3,000–8,000 行**（視 country 數量而定）

***

### View 2：每月 × 渠道摘要（用於 Page 3 趨勢）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_monthly_summary` AS

SELECT
  FORMAT_DATE('%Y-%m', session_date)   AS year_month,
  channel,
  device,
  COUNT(*)                             AS total_sessions,
  COUNTIF(is_bounce = 1)              AS bounced_sessions,
  ROUND(AVG(session_duration_sec), 1) AS avg_duration_sec,
  ROUND(AVG(pages_viewed), 2)         AS avg_pages_viewed,
  COUNTIF(engagement_tier = 'High')   AS high_engagement_sessions
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY year_month, channel, device;



> 511,797 行 → 約 **200–400 行**

***

### View 3：國家 × 渠道摘要（用於地圖/國家分析）


In [ ]:

CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_country_summary` AS

SELECT
  country,
  channel,
  device,
  COUNT(*)                             AS total_sessions,
  COUNTIF(is_bounce = 1)              AS bounced_sessions,
  ROUND(AVG(session_duration_sec), 1) AS avg_duration_sec
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY country, channel, device;


> 511,797 行 → 約 **100–300 行**

***

## 行數預估對比

| 表/View | 原始行數 | Import 後估計行數 | 壓縮率 |
|---|---|---|---|
| `sessions`（原始） | 511,797 | 511,797 | — |
| `v_sessions_daily_summary` | → | ~5,000 | ~99% |
| `v_sessions_monthly_summary` | → | ~300 | ~99.9% |
| `v_sessions_country_summary` | → | ~200 | ~99.9% |

***

## 執行建議

1. 在 BigQuery Console 逐條跑上面 3 條 SQL 建立 View
2. 確認 row count：

In [ ]:
SELECT 'v_sessions_daily_summary',   COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_daily_summary`
UNION ALL
SELECT 'v_sessions_monthly_summary', COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_monthly_summary`
UNION ALL
SELECT 'v_sessions_country_summary', COUNT(*) FROM `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_country_summary`;


3. 在 Power BI **import 這 3 個 view** 取代原始 `sessions` 表
4. 原始 `sessions` 完全不需要 import 到 Power BI


3 個聚合 View 都成功建立，行數非常理想。

Row Count 結果

| View                       | 行數     | 壓縮率      |
| -------------------------- | ------ | -------- |
| v_sessions_daily_summary   | 82,077 | 84% ⬇️   |
| v_sessions_monthly_summary | 180    | 99.97% ⬇️  |
| v_sessions_country_summary | 120    | 99.98% ⬇️  |

如果想更輕量，可以去掉 country 和 engagement_tier 維度，  
去掉後預計約 3,000–5,000 行，country 分析由 v_sessions_country_summary（120行）負責。：

In [ ]:
CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.v_sessions_daily_summary` AS

SELECT
  session_date,
  channel,
  device,
  COUNT(*)                              AS total_sessions,
  COUNTIF(is_bounce = 1)               AS bounced_sessions,
  SUM(pages_viewed)                    AS total_pages_viewed,
  SUM(session_duration_sec)            AS total_duration_sec,
  COUNTIF(engagement_tier = 'High')    AS high_engagement_sessions,
  COUNTIF(engagement_tier = 'Medium')  AS medium_engagement_sessions,
  COUNTIF(engagement_tier = 'Low')     AS low_engagement_sessions
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
GROUP BY session_date, channel, device;


考慮Power BI Desktop Import 模式：官方建議單表不超過 3,000 萬行，   
現行 82,077 行不需要再壓縮，開始下一步。 

## 建立 campaigns (channel) 關聯的問題，都是 Many-to-Many。  
這是因為 campaigns[channel] 有重複值（例如多個 Google Ads 活動），連接到 view 的 channel 也有重複值。  
雙向篩選風險，可能造成數字重複計算。  

解決方法  
要求: 源頭新增任何渠道，view 自動包含，日後新增修改維護方便。  
 建立一張獨立的 mapping 表，與動態 view 分離：

In [ ]:
-- ============================================================
-- 建 dim_channel_mapping 之前的預備檢查
-- ============================================================

-- 1. 查看 campaigns 現有所有唯一 channel（確認有哪些值需要加入 mapping）
SELECT DISTINCT channel, COUNT(*) AS campaign_count
FROM `ross-bi-project-03.traffic_ad_roi_clean.campaigns`
WHERE channel IS NOT NULL
GROUP BY channel
ORDER BY channel;


-- 2. 跨表確認：所有表的 channel 值是否一致
SELECT 'campaigns'              AS source, channel FROM `ross-bi-project-03.traffic_ad_roi_clean.campaigns`
UNION DISTINCT
SELECT 'conversions',             channel FROM `ross-bi-project-03.traffic_ad_roi_clean.conversions`
UNION DISTINCT
SELECT 'sessions',                channel FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
UNION DISTINCT
SELECT 'v_channel_performance',   channel FROM `ross-bi-project-03.traffic_ad_roi_clean.v_channel_performance`
ORDER BY channel;


-- 3. 找出各表有但 campaigns 沒有的 channel（孤兒值）
SELECT 'conversions' AS source, channel
FROM `ross-bi-project-03.traffic_ad_roi_clean.conversions`
WHERE channel NOT IN (SELECT DISTINCT channel FROM `ross-bi-project-03.traffic_ad_roi_clean.campaigns`)
UNION DISTINCT
SELECT 'sessions', channel
FROM `ross-bi-project-03.traffic_ad_roi_clean.sessions`
WHERE channel NOT IN (SELECT DISTINCT channel FROM `ross-bi-project-03.traffic_ad_roi_clean.campaigns`);


In [ ]:
-- Step 1：建 mapping 表（只維護 sort_order）
CREATE OR REPLACE TABLE `ross-bi-project-03.traffic_ad_roi_clean.dim_channel_mapping` AS
SELECT * FROM UNNEST([
  STRUCT('Google Ads'   AS channel, 1 AS sort_order),
  STRUCT('Facebook Ads' AS channel, 2 AS sort_order),
  STRUCT('Email'        AS channel, 3 AS sort_order),
  STRUCT('Organic'      AS channel, 4 AS sort_order),
  STRUCT('Direct'       AS channel, 5 AS sort_order)
]);


-- Step 2：動態 view，新渠道自動包含，sort_order 預設 99
CREATE OR REPLACE VIEW `ross-bi-project-03.traffic_ad_roi_clean.dim_channel` AS

SELECT
  c.channel,
  COALESCE(m.sort_order, 99) AS sort_order
FROM (
  SELECT DISTINCT channel
  FROM `ross-bi-project-03.traffic_ad_roi_clean.campaigns`
  WHERE channel IS NOT NULL
) c
LEFT JOIN `ross-bi-project-03.traffic_ad_roi_clean.dim_channel_mapping` m
  USING (channel)
ORDER BY sort_order;


核心 DAX Measures

In [ ]:
-- 收入 & 花費
Total Revenue USD   = SUM(conversions[order_value_usd])
Total Spend USD     = SUM(ad_impressions[spend_usd])
Total Orders        = COUNTROWS(conversions)
Total Impressions   = SUM(ad_impressions[impressions])
Total Clicks        = SUM(ad_impressions[clicks])
Total Sessions      = SUM(v_sessions_daily_summary[total_sessions])

-- 效率指標
ROAS                = DIVIDE([Total Revenue USD], [Total Spend USD], 0)
ROI %               = DIVIDE([Total Revenue USD] - [Total Spend USD], [Total Spend USD], 0)
CTR %               = DIVIDE([Total Clicks], [Total Impressions], 0)
CVR %               = DIVIDE([Total Orders], [Total Sessions], 0)
CPA                 = DIVIDE([Total Spend USD], [Total Orders], 0)
AOV                 = DIVIDE([Total Revenue USD], [Total Orders], 0)
Bounce Rate         = DIVIDE(SUM(v_sessions_daily_summary[bounced_sessions]), [Total Sessions], 0)

-- 條件格式用（標示虧損）
ROAS Color          = IF([ROAS] >= 2, "#27AE60", IF([ROAS] >= 1, "#F39C12", "#E74C3C"))


ERD

In [ ]:
erDiagram
    Date_Table {
        date Date PK
        int Year
        int Month
        string MonthName
        string Quarter
        string YearMonth
        string WeekDay
    }

    dim_channel {
        string channel PK
        int sort_order
    }

    campaigns {
        string campaign_id PK
        string campaign_name
        string channel FK
        string campaign_type
        float daily_budget_usd
        date start_date
        date end_date
        bool is_active
    }

    ad_impressions {
        string impression_id PK
        string campaign_id FK
        date date FK
        int impressions
        int clicks
        float ctr_calculated
        float ctr_original
        float spend_usd
        float cost_per_click_usd
    }

    conversions {
        string order_id PK
        string session_id
        string campaign_id FK
        string channel
        date order_date FK
        float order_value_usd
        string order_value_tier
        string device
        string country
    }

    v_campaign_roi {
        string campaign_id FK
        string campaign_name
        string channel
        float total_spend_usd
        float total_revenue_usd
        float roas
        float roi_pct
        float cpa
        float ctr_pct
        float session_cvr_pct
    }

    v_campaign_ctr_cvr_scatter {
        string campaign_id FK
        string campaign_name
        float avg_ctr_pct
        float avg_session_cvr_pct
        float avg_click_cvr_pct
        float total_revenue_usd
        float total_spend_usd
    }

    v_campaign_daily_ctr_cvr {
        string campaign_id FK
        date date FK
        float ctr
        float session_cvr
        float click_cvr
        int impressions
        int clicks
        float spend_usd
    }

    v_channel_performance {
        string channel FK
        int total_sessions
        float bounce_rate
        int total_orders
        float total_revenue_usd
        float roas
        float conversion_rate
        float cost_per_acquisition
    }

    v_device_channel_conversion {
        string channel FK
        string device
        int orders
        float revenue_usd
        float avg_order_value
    }

    v_monthly_roi_trend {
        string year_month
        string channel FK
        float spend_usd
        float revenue_usd
        float roas
        float roi_pct
    }

    v_monthly_channel_trend {
        string year_month
        string channel FK
        int orders
        float revenue_usd
    }

    v_sessions_daily_summary {
        date session_date FK
        string channel FK
        string device
        string country
        int total_sessions
        int bounced_sessions
        int high_engagement_sessions
    }

    v_sessions_monthly_summary {
        string year_month
        string channel FK
        string device
        int total_sessions
        float avg_duration_sec
    }

    v_sessions_country_summary {
        string country
        string channel FK
        string device
        int total_sessions
        float avg_duration_sec
    }

    v_campaign_type_roi {
        string channel
        string campaign_type
        float avg_roas
        float avg_roi_pct
        float avg_cpa_usd
    }

    v_ctr_bucket_analysis {
        string ctr_bucket
        int record_count
        float avg_ctr_pct
        float avg_session_cvr_pct
        float total_revenue_usd
    }

    dim_channel         ||--o{ campaigns                  : "channel"
    dim_channel         ||--o{ v_channel_performance       : "channel"
    dim_channel         ||--o{ v_device_channel_conversion : "channel"
    dim_channel         ||--o{ v_monthly_roi_trend         : "channel"
    dim_channel         ||--o{ v_monthly_channel_trend     : "channel"
    dim_channel         ||--o{ v_sessions_daily_summary    : "channel"
    dim_channel         ||--o{ v_sessions_monthly_summary  : "channel"
    dim_channel         ||--o{ v_sessions_country_summary  : "channel"

    campaigns           ||--o{ ad_impressions              : "campaign_id"
    campaigns           ||--o{ conversions                 : "campaign_id"
    campaigns           ||--o{ v_campaign_roi              : "campaign_id"
    campaigns           ||--o{ v_campaign_ctr_cvr_scatter  : "campaign_id"
    campaigns           ||--o{ v_campaign_daily_ctr_cvr    : "campaign_id"

    Date_Table          ||--o{ ad_impressions              : "date"
    Date_Table          ||--o{ conversions                 : "order_date"
    Date_Table          ||--o{ v_campaign_daily_ctr_cvr    : "date"
    Date_Table          ||--o{ v_sessions_daily_summary    : "session_date"
